In [1]:
import torch

print("PyTorch Version:", torch.__version__)
print("Is GPU available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

PyTorch Version: 2.14.1+cu126
Is GPU available?: True
GPU Name: NVIDIA GeForce RTX 3070 Ti Laptop GPU


In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

text = "Hello, world! Byte distillation is"
inputs = tokenizer(text, return_tensors="pt")

# inspect how BPE splits words like "distillation"
tokens = [tokenizer.decode([tid]) for tid in inputs["input_ids"][0]]
print("Token IDs:", inputs["input_ids"].tolist())
print("Tokens:", tokens)
print("Vocab size:", tokenizer.vocab_size)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Token IDs: [[15496, 11, 995, 0, 30589, 1233, 40903, 318]]
Tokens: ['Hello', ',', ' world', '!', ' Byte', ' dist', 'illation', ' is']
Vocab size: 50257


In [5]:
with torch.no_grad():
    outputs = model(**inputs)

# outputs.logits is [batch, seq_len, vocab_size] -> take last position
next_token_logits = outputs.logits[0, -1, :]
next_token_probs = F.softmax(next_token_logits, dim=-1)

top_probs, top_ids = torch.topk(next_token_probs, 5)
for prob, tid in zip(top_probs, top_ids):
    print(f"{repr(tokenizer.decode([tid]))}: {prob.item():.4f}")

' a': 0.1026
' the': 0.0477
' not': 0.0373
' here': 0.0361
' now': 0.0277
